In [12]:
import pandas as pd
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [3]:
from pathlib import Path

DATA_PATH = Path("../data/vgchartz-2024.csv")
FIGURES_DIR = Path("figures")

In [4]:

import os
#print(os.getcwd())

def load_dataset(path: Path) -> pd.DataFrame:
    """Load the raw CSV dataset."""
    return pd.read_csv(path)

def missing_summary(df: pd.DataFrame) -> pd.DataFrame:
    """Return missing-value counts and percentages."""
    summary = pd.DataFrame(
        {
            "missing_count": df.isna().sum(),
            "missing_pct": (df.isna().mean() * 100).round(2),
        }
    )
    return summary.sort_values("missing_count", ascending=False)

df = load_dataset(DATA_PATH)
FIGURES_DIR.mkdir(exist_ok=True)

print(f"Dataset shape: {df.shape}")
print("\nColumn types:")
print(df.dtypes)
print(missing_summary(df))

# Drop all rows with missing values
new_df = df.copy()
print(missing_summary(new_df))

new_df["critic_score_missing"] = new_df["critic_score"].isna().astype(int)
new_df["critic_score"] = new_df["critic_score"].fillna(new_df["critic_score"].median())

#filling in missing values for categorical data. "Unknown" could hold some significance in the data
new_df["genre"] = new_df["genre"].fillna("Unknown")
new_df["publisher"] = new_df["publisher"].fillna("Unknown")
new_df["developer"] = new_df["developer"].fillna("Unknown")

#reducing cardinality in publisher and developer
top_publishers = new_df["publisher"].value_counts().nlargest(40).index
new_df["publisher"] = new_df["publisher"].where(new_df["publisher"].isin(top_publishers), "Other")

top_developers = new_df["developer"].value_counts().nlargest(40).index
new_df["developer"] = new_df["developer"].where(new_df["developer"].isin(top_developers), "Other")



# dropping all regional sales to avoid data leakage
X = new_df.drop(columns=["release_date", "console", "developer", "publisher", "img", "title", "last_update", "total_sales", "na_sales", "jp_sales", "pal_sales", "other_sales"])

#one-hot encoding to change categorical features to numerical:
X = pd.get_dummies(X, columns = ["genre"], drop_first = True)

y = new_df["total_sales"].copy()



Dataset shape: (64016, 14)

Column types:
img                 str
title               str
console             str
genre               str
publisher           str
developer           str
critic_score    float64
total_sales     float64
na_sales        float64
jp_sales        float64
pal_sales       float64
other_sales     float64
release_date        str
last_update         str
dtype: object
              missing_count  missing_pct
critic_score          57338        89.57
jp_sales              57290        89.49
na_sales              51379        80.26
pal_sales             51192        79.97
other_sales           48888        76.37
last_update           46137        72.07
total_sales           45094        70.44
release_date           7051        11.01
developer                17         0.03
publisher                 0         0.00
img                       0         0.00
genre                     0         0.00
console                   0         0.00
title                     0       

In [5]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

def split_data(X, y, test_size=0.2, random_state=42):
    """
    Splits the data into training and test sets.
    """
    return train_test_split(X, y, test_size=test_size, random_state=random_state)

def train_model(X_train, y_train):
    """
    Trains a Logistic Regression classifier.
    Returns:
        model: trained scikit-learn classifier
    """
    model = LinearRegression()
    model.fit(X_train, y_train)
    
    return model

In [6]:
# remove rows where target is missing

mask = new_df["total_sales"].notna()

X = X.loc[mask].copy()
y = new_df.loc[mask, "total_sales"].copy()

In [7]:

X_train, X_test, y_train, y_test = split_data(X, y, test_size=0.2, random_state=42)

model = train_model(X_train, y_train)

In [8]:
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np
def evaluate_model(model, X_test, y_test):
    """
    Evaluates the trained model.
    Returns: MSE: mean squared error: penalizes large errors 
    RMSE: makes the error interpretable as it shows the error in the same units as target val. 
    R^2: explains how the well model predicted (explains variance)
        
    """
    y_pred = model.predict(X_test)
    
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, y_pred)

    return{"MSE: ": mse, "RMSE: ": rmse, "R^2: ": r2}

metrics = evaluate_model(model, X_test, y_test)
metrics



{'MSE: ': 0.6361715055606078,
 'RMSE: ': np.float64(0.7976036017725896),
 'R^2: ': 0.13012576193877268}

This is here for documentation for presentation purposes to show our process:

Iteration 1:

We are using MSE, RMSE because the error values are continuous so accuracy and precision don't work (they only work on categorical data)

Accuracy and Precision metrics had given us an error previously

The returned values of:
{'MSE: ': 4.7464313888453375e-05,
 'RMSE: ': 0.006889434946964328,
 'R^2: ': 0.9999910678863546}

are suspiciously good. R^2 is 1 - error, meaning that there was almost none and then the RMSE shows that the eror is less than 1 hundredth off 

this tells us that theres a chance we have "leakage" somewhere or our data is too simple

***We should also consider using Log the data as some blockbuster games are huge but most don't sell as much so we should test on both this way and with 

We should also use one-hot encoding for the categorical data such as developer, genre, etc because they are features which are important but are categorical

Iteration 2:

we removed any features that may have been leading to leakage such as: {na_sales, jp_sales, other_sales} so the model didn't see the answer indirectly

We flagged any missing critic scores and then imputed the median because it is less sensitive to skewed data than the mean. By doing this we flag any missing critic score and still give it a value so it can be used by the model
We marked the categorical features that had missing cells with "Unknown" because like with the critic score this missing data could help the model predict more accurately

We also sstarted using one-hot encoding for genre and will see if using it for publisher and developer works as well
We made publisher andd developer have less cardinality by selecting only the top 40 respectively
Below we added a function that will determine if a feature is useful or not

{'MSE: ': 0.6361715055606074,
 'RMSE: ': 0.7976036017725894,
 'R^2: ': 0.13012576193877323}

In [10]:
# Evaluating usefulness of features

def run_experiment(X, y, featuresToDrop=None):
    if featuresToDrop:
        X_exp = X.drop(columns = featuresToDrop)
    else:
        X_exp = X.copy()

    X_train, X_test, y_train, y_test = train_test_split(X_exp, y, test_size = 0.2, random_state = 42)
    
    model = LinearRegression()
    model.fit(X_train, y_train)
    
    metrics = evaluate_model(model, X_test, y_test)
    return metrics

#list of features to be evaluated needs to be added: "drop_developer": ["developer"], "drop_critic_score": ["critic_sccore"], "drop_release_date": ["release_date"],"drop_critic_score": ["critic_sccore"], "drop_release_date": ["release_date"],  "drop_publisher": ["publisher"]
experiments = {"baseline ": [],  
               "drop_genre": ["genre"]}

# list that will hold the results
results = {}

for name, features in experiments.items():
    print(f"\n Running experiment: {name}")
    
    metrics = run_experiment(X, y, features)
    
    results[name] = metrics
    print(metrics)
    
#summary table
import pandas as pd
summary = pd.DataFrame(results).T
print("\n Summary of results: ")
print(summary)


 Running experiment: baseline 
{'MSE: ': 0.6361715055606078, 'RMSE: ': np.float64(0.7976036017725896), 'R^2: ': 0.13012576193877268}

 Running experiment: drop_genre


KeyError: "['genre'] not found in axis"

In [20]:
# TEST 2: Log-Transformed Target Experiment

# The target (total_sales) is highly skewed.
# We apply a log transformation to reduce the effect of extreme outliers
# and test whether this improves the model's performance.

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np
import pandas as pd

# helper function
def get_metrics(y_true, y_pred):
    return {
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "MAE": mean_absolute_error(y_true, y_pred),
        "R2": r2_score(y_true, y_pred),
    }

# 1. baseline metrics
baseline_preds = model.predict(X_test)
baseline_metrics = get_metrics(y_test, baseline_preds)

# 2. transform target
y_log = np.log1p(y)

# 3. split data
X_train_log, X_test_log, y_train_log, y_test_log = train_test_split(
    X, y_log, test_size=0.2, random_state=42
)

# 4. train model on transformed target
model_log = LinearRegression()
model_log.fit(X_train_log, y_train_log)

# 5. predict and convert back to original scale
log_preds = model_log.predict(X_test_log)
preds_log = np.expm1(log_preds)
actuals_log = np.expm1(y_test_log)

# 6. evaluate log model on original scale
metrics_log = get_metrics(actuals_log, preds_log)

# 7. clean comparison table
comparison_df = pd.DataFrame(
    [baseline_metrics, metrics_log],
    index=["Baseline", "Log Model"]
).round(4)

print("\nModel Comparison")
print(comparison_df)



Model Comparison
             RMSE     MAE      R2
Baseline   0.7976  0.3475  0.1301
Log Model  0.8070  0.3087  0.1095


Test 2: Log-Transformed Target Experiment

Due to the total_sales being mainly right-skewed, we tested a log transformation to see if it would reduce the influence of major blockbuster games. However, the transformation-target regressions did not improve the overall performance. RMSE increased slightly from 0.7976 to 0.8070, and R^2 decreased from 0.1301 to 0.1095. These findings suggest that the current feature set and linear regression set up, log-transforming the target did not improve the output. Therefore keeping the original target scale as the stronger baseline.